In [1]:
# Cell 1: Setup & Configuration
import os
import re
import io
import requests
import zipfile
import pandas as pd
import numpy as np
import logging
from datetime import datetime, timedelta
from pandas_datareader import data as pdr

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- PIPELINE CONFIGURATION ---

# 1. Date Range for the Pipeline
START_DATE = '2025-12-18' 
END_DATE   = '2025-12-20'

# 2. Target Stocks & Strict Aliases
# Logic: We ONLY keep news where the Title/URL explicitly mentions these keywords.
STOCK_CONFIG = {
    # 1. THE HYPE STOCK: NVIDIA
    # High Volatility, High Sentiment Sensitivity
    'NVDA': {
        'aliases': ['nvidia', 'jensen huang', 'geforce', 'rtx', 'nvda'],
        'blacklist': [] # "Nvidia" is unique, very little noise
    },
    
    # 2. THE STABLE STOCK: MICROSOFT
    # Lower Volatility, "Control Group"
    'MSFT': {
        'aliases': ['microsoft', 'satya nadella', 'azure', 'msft'],
        'blacklist': [] # "Microsoft" is unique
    }
}

# 3. Layer Paths (Simulating S3/MinIO Buckets)
BASE_DIR = './data_pipeline'
BRONZE_DIR = f'{BASE_DIR}/bronze'   # Raw Data
SILVER_DIR = f'{BASE_DIR}/silver'   # Cleaned & Filtered
GOLD_DIR   = f'{BASE_DIR}/gold'     # Aggregated & ML Ready

for d in [BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(d, exist_ok=True)
    
print("✓ Configuration Loaded & Directories Created")


✓ Configuration Loaded & Directories Created


In [2]:
# Cell 2: BRONZE LAYER - Ingestion (BigQuery)
# Goal: Fetch ONLY relevant rows (English Only) + Get Extras for better titles.

from google.cloud import bigquery
import os
import re
import logging
from datetime import datetime, timedelta
from pandas_datareader import data as pdr

# --- DATE CONFIGURATION (USE REAL DATA) ---
# Use "Today minus X days" to ensure data exists in the real GDELT table
TODAY = datetime.now()
START_DATE = (TODAY - timedelta(days=3)).strftime('%Y-%m-%d')
END_DATE = (TODAY - timedelta(days=1)).strftime('%Y-%m-%d')

logger.info(f"Querying Real-World GDELT Data: {START_DATE} to {END_DATE}")

def generate_bq_filter(config):
    """
    Constructs a SQL WHERE clause.
    """
    stock_conditions = []
    
    for ticker, rules in config.items():
        # FIX: Use single backslash logic for f-strings to get \b in Regex
        # f"\\b" -> string "\b" -> BQ Regex "Word Boundary"
        aliases = [f"\\b{re.escape(a)}\\b" for a in rules['aliases']]
        alias_str = "|".join(aliases)
        
        condition = f"REGEXP_CONTAINS(DocumentIdentifier, r'(?i){alias_str}')"
        
        if rules['blacklist']:
            blacklist = [re.escape(b) for b in rules['blacklist']]
            blk_str = "|".join(blacklist)
            condition += f" AND NOT REGEXP_CONTAINS(DocumentIdentifier, r'(?i){blk_str}')"
            
        stock_conditions.append(f"({condition})")
    
    return " OR ".join(stock_conditions)

def download_gdelt_bronze_bq(start, end, output_dir, config):
    client = bigquery.Client()
    filter_logic = generate_bq_filter(config)
    
    # UPGRADE:
    # 1. Select 'Extras' to get the real Page Title.
    # 2. Select 'TranslationInfo' to check language.
    # 3. WHERE clause: Ensure TranslationInfo IS NULL (Native English).
    query = f"""
        SELECT 
            DATE, 
            DocumentIdentifier as source_url, 
            Themes as themes, 
            Organizations as organizations, 
            V2Tone as tone,
            Extras as extras,
            TranslationInfo as translation_info
        FROM `gdelt-bq.gdeltv2.gkg_partitioned`
        WHERE _PARTITIONDATE BETWEEN DATE('{start}') AND DATE('{end}')
          AND (TranslationInfo IS NULL OR LENGTH(TranslationInfo) = 0) -- English Only Filter
          AND ({filter_logic})
    """
    
    logger.info("Running BigQuery Job...")
    
    try:
        query_job = client.query(query)
        df = query_job.to_dataframe()
        
        if not df.empty:
            filename = f"gdelt_bronze_bq_{start}_{end}.csv"
            save_path = os.path.join(output_dir, filename)
            df.to_csv(save_path, index=False)
            logger.info(f"✓ Downloaded {len(df)} English-only rows to {save_path}")
            return save_path
        else:
            logger.warning(f"BigQuery returned 0 rows for {start} to {end}.")
            return None
            
    except Exception as e:
        logger.error(f"BigQuery Failed: {e}")
        return None

def download_stooq_bronze(tickers, output_dir):
    for ticker in tickers:
        logger.info(f"Downloading Bronze Stooq: {ticker}")
        try:
            # Fetch last 5 years to be safe
            df = pdr.DataReader(ticker, 'stooq', start=datetime(2020, 1, 1), end=datetime.now())
            if not df.empty:
                save_path = f"{output_dir}/stooq_{ticker}.csv"
                df.to_csv(save_path)
        except Exception as e:
            logger.error(f"Stooq failed for {ticker}: {e}")

# --- EXECUTION ---
print("--- Starting Bronze Ingestion (BigQuery) ---")
bq_file = download_gdelt_bronze_bq(START_DATE, END_DATE, BRONZE_DIR, STOCK_CONFIG)
download_stooq_bronze(STOCK_CONFIG.keys(), BRONZE_DIR)
print("✓ Bronze Layer Complete")

2025-12-22 16:49:18,478 - INFO - Querying Real-World GDELT Data: 2025-12-19 to 2025-12-21
/Users/mac/Git/DQOPs/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


--- Starting Bronze Ingestion (BigQuery) ---


/Users/mac/Git/DQOPs/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
2025-12-22 16:49:19,757 - INFO - Running BigQuery Job...
/Users/mac/Git/DQOPs/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2025-12-22 16:49:21,935 - INFO - ✓ Downloaded 332 English-only rows to ./data_pipeline/bronze/gdelt_bronze_bq_2025-12-19_2025-12-21.csv
2025-12-22 16:49:21,936 - INFO - Downloading Bronze Stooq: NVDA
2025-12-22 16:49:23,508 - INFO - Downloading Bronze Stooq: MSFT


✓ Bronze Layer Complete


In [3]:
# Cell 3: SILVER LAYER - Transformation (Strict Title Gatekeeping)
# Goal: Read BQ Dump, Extract Best Title, and DROP any row where the Title 
# does not explicitly mention the Alias.

def extract_title_from_xml(extras_str):
    """
    Parses <PAGE_TITLE>My Title</PAGE_TITLE> from the Extras column.
    """
    if not isinstance(extras_str, str): return None
    try:
        # Regex to find content inside <PAGE_TITLE> tags
        match = re.search(r'<PAGE_TITLE>(.*?)</PAGE_TITLE>', extras_str)
        if match:
            return match.group(1).strip()
    except:
        pass
    return None

def extract_title_from_url_slug(url):
    """Fallback: Extracts title from URL slug."""
    if not isinstance(url, str): return ""
    try:
        slug = re.search(r'([^/]+)(?:\.html|\?|$)', url.strip('/'))
        if slug:
            text = slug.group(1)
            # Remove digits, IDs, replace separators
            text = re.sub(r'\d+', '', text)
            text = text.replace('-', ' ').replace('_', ' ').replace('+', ' ')
            return text.strip().title()
    except:
        pass
    return ""

def process_gdelt_silver(bronze_dir, silver_dir, config):
    # Find the BigQuery dump file
    all_files = [f for f in os.listdir(bronze_dir) if f.startswith('gdelt_bronze_bq_')]
    if not all_files:
        logger.warning("No BigQuery dump found in Bronze.")
        return pd.DataFrame()
    
    silver_rows = []
    
    for f in all_files:
        path = os.path.join(bronze_dir, f)
        try:
            # Load Bronze Data
            df = pd.read_csv(path)
        except Exception as e:
            logger.warning(f"Could not read {f}: {e}")
            continue

        # --- STEP 1: RESOLVE TITLES ---
        # Try XML first, fallback to URL
        df['xml_title'] = df['extras'].apply(extract_title_from_xml)
        df['url_title'] = df['source_url'].apply(extract_title_from_url_slug)
        df['final_title'] = df['xml_title'].fillna(df['url_title'])
        
        # Basic cleanup
        df = df[df['final_title'].str.len() > 10]

        # --- STEP 2: STRICT FILTERING ---
        for ticker, rules in config.items():
            aliases = rules['aliases']
            
            # Create Regex Pattern with Word Boundaries
            boundary_aliases = [r'\b' + re.escape(a) + r'\b' for a in aliases]
            alias_pattern = '|'.join(boundary_aliases)
            
            # STRICT RULE: The *Title* itself must contain the alias.
            # We NO LONGER check the URL for the filter. If the title is "Are you a human?",
            # it fails this check even if the URL has "nvidia".
            mask_title = df['final_title'].str.contains(alias_pattern, case=False, na=False, regex=True)
            
            # Apply Blacklist to BOTH Title and URL to be safe
            mask_noise = pd.Series([False] * len(df), index=df.index)
            if rules['blacklist']:
                blk_pattern = '|'.join([re.escape(b) for b in rules['blacklist']])
                # Check Title for noise
                noise_title = df['final_title'].str.contains(blk_pattern, case=False, na=False, regex=True)
                # Check URL for noise
                noise_url = df['source_url'].str.contains(blk_pattern, case=False, na=False, regex=True)
                mask_noise = noise_title | noise_url

            # Final Filter: Must match Title AND NOT match Blacklist
            matches = df[mask_title & (~mask_noise)].copy()
            
            if not matches.empty:
                matches['ticker'] = ticker
                matches['date'] = pd.to_datetime(matches['DATE'].astype(str).str[:8], format='%Y%m%d')
                
                # Select final clean columns
                silver_rows.append(matches[['date', 'ticker', 'final_title', 'source_url']].rename(columns={'final_title': 'extracted_title'}))
                
    if silver_rows:
        df_silver = pd.concat(silver_rows, ignore_index=True)
        # Deduplicate
        df_silver = df_silver.drop_duplicates(subset=['date', 'ticker', 'extracted_title'])
        
        out_path = f"{silver_dir}/news_silver.csv"
        df_silver.to_csv(out_path, index=False)
        print(f"Saved Silver News: {out_path} ({len(df_silver)} rows)")
        return df_silver
    else:
        print("No relevant news found in Bronze.")
        return pd.DataFrame()

def process_stooq_silver(bronze_dir, silver_dir):
    # (Unchanged)
    all_files = [f for f in os.listdir(bronze_dir) if f.startswith('stooq_')]
    silver_stocks = []
    
    for f in all_files:
        ticker = f.replace('stooq_', '').replace('.csv', '')
        df = pd.read_csv(os.path.join(bronze_dir, f))
        
        df.columns = [c.lower() for c in df.columns]
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
            df['ticker'] = ticker
            silver_stocks.append(df[['date', 'ticker', 'close', 'open', 'high', 'low', 'volume']])
            
    if silver_stocks:
        df_stocks = pd.concat(silver_stocks, ignore_index=True)
        out_path = f"{silver_dir}/stocks_silver.csv"
        df_stocks.to_csv(out_path, index=False)
        print(f"Saved Silver Stocks: {out_path}")

# --- EXECUTION ---
print("--- Starting Silver Transformation ---")
df_news_silver = process_gdelt_silver(BRONZE_DIR, SILVER_DIR, STOCK_CONFIG)
process_stooq_silver(BRONZE_DIR, SILVER_DIR)
print("✓ Silver Layer Complete")

--- Starting Silver Transformation ---
Saved Silver News: ./data_pipeline/silver/news_silver.csv (252 rows)
Saved Silver Stocks: ./data_pipeline/silver/stocks_silver.csv
✓ Silver Layer Complete


In [4]:
# Cell 4: NLP Processing
# Goal: Prepare data for model, simulate scoring, and get ready for Gold aggregation.

def generate_nlp_input(silver_dir):
    """Creates the file your NLP model needs to read."""
    path = f"{silver_dir}/news_silver.csv"
    if not os.path.exists(path): return pd.DataFrame()
    
    df = pd.read_csv(path)
    # The NLP model needs strict columns: date, entity, text
    nlp_input = df[['date', 'ticker', 'extracted_title']].rename(columns={'ticker': 'entity', 'extracted_title': 'title'})
    
    # Save input for external model
    nlp_input.to_csv(f"{silver_dir}/nlp_input_ready.csv", index=False)
    print(f"Generated NLP Input: {silver_dir}/nlp_input_ready.csv")
    return nlp_input

def simulate_nlp_output(silver_dir):
    """
    MOCKS the output of your T5/BERT model. 
    In production, you would run the model script here instead.
    """
    input_df = generate_nlp_input(silver_dir)
    if input_df.empty: return pd.DataFrame()
    
    # Mocking a Sentiment Score (-1 to 1)
    # We add random noise to test the pipeline
    scored_df = input_df.copy()
    scored_df['sentiment_score'] = np.random.uniform(-1, 1, size=len(scored_df))
    
    # Save as if the model just finished running
    scored_df.to_csv(f"{silver_dir}/nlp_output_scored.csv", index=False)
    print(f"Simulated NLP Output: {silver_dir}/nlp_output_scored.csv")
    return scored_df

# --- EXECUTION ---
print("--- Starting NLP Processing ---")
df_scored_news = simulate_nlp_output(SILVER_DIR)


--- Starting NLP Processing ---
Generated NLP Input: ./data_pipeline/silver/nlp_input_ready.csv
Simulated NLP Output: ./data_pipeline/silver/nlp_output_scored.csv


In [5]:
# Cell 5: GOLD LAYER - Aggregation & ML Features
# Goal: Join Stocks + Sentiment, Create Lags, Target Variables.

def create_gold_layer(silver_dir, gold_dir):
    # 1. Load Data
    try:
        stocks = pd.read_csv(f"{silver_dir}/stocks_silver.csv")
        news = pd.read_csv(f"{silver_dir}/nlp_output_scored.csv")
        stocks['date'] = pd.to_datetime(stocks['date'])
        news['date'] = pd.to_datetime(news['date'])
    except Exception as e:
        logger.error(f"Missing Silver data: {e}")
        return

    # 2. Aggregate News (Many articles per day -> One score per day)
    # Logic: Weighted Average could be better, but we start with Mean.
    daily_sentiment = news.groupby(['date', 'entity']).agg({
        'sentiment_score': 'mean',
        'title': 'count'
    }).reset_index().rename(columns={'entity': 'ticker', 'sentiment_score': 'sentiment_avg', 'title': 'news_volume'})

    # 3. Merge (Left Join on Stocks to keep market days)
    gold_df = pd.merge(stocks, daily_sentiment, on=['date', 'ticker'], how='left')
    
    # Fill missing news with Neutral (0)
    gold_df['sentiment_avg'] = gold_df['sentiment_avg'].fillna(0)
    gold_df['news_volume'] = gold_df['news_volume'].fillna(0)

    # 4. Feature Engineering (The "Business Analytics" Value)
    gold_df = gold_df.sort_values(['ticker', 'date'])
    
    gold_dfs = []
    for ticker in gold_df['ticker'].unique():
        group = gold_df[gold_df['ticker'] == ticker].copy()
        
        # Lag 1: Yesterday's Sentiment predicts Today
        group['lag1_sentiment'] = group['sentiment_avg'].shift(1)
        
        # Moving Averages
        group['ma_7'] = group['close'].rolling(7).mean()
        group['ma_30'] = group['close'].rolling(30).mean()
        
        # Target: Next Day Return (for Prediction)
        group['daily_return'] = group['close'].pct_change()
        group['target_next_return'] = group['daily_return'].shift(-1)
        
        gold_dfs.append(group)
        
    final_gold = pd.concat(gold_dfs)
    
    # 5. Save Serving Tables
    # A. For Machine Learning (Drop rows with NaN targets)
    ml_data = final_gold.dropna()
    ml_data.to_csv(f"{gold_dir}/ml_training_data.csv", index=False)
    
    # B. For Dashboard (Keep everything to show gaps)
    final_gold.to_csv(f"{gold_dir}/superset_dashboard_data.csv", index=False)
    
    print(f"✓ Gold Layer Complete.")
    print(f"  - ML Data: {gold_dir}/ml_training_data.csv")
    print(f"  - Dashboard Data: {gold_dir}/superset_dashboard_data.csv")
    print("\nSample Data:")
    print(final_gold[['date', 'ticker', 'close', 'sentiment_avg', 'lag1_sentiment']].tail())

# --- EXECUTION ---
print("--- Starting Gold Aggregation ---")
create_gold_layer(SILVER_DIR, GOLD_DIR)

--- Starting Gold Aggregation ---
✓ Gold Layer Complete.
  - ML Data: ./data_pipeline/gold/ml_training_data.csv
  - Dashboard Data: ./data_pipeline/gold/superset_dashboard_data.csv

Sample Data:
        date ticker   close  sentiment_avg  lag1_sentiment
4 2025-12-15   NVDA  176.29        0.00000             0.0
3 2025-12-16   NVDA  177.72        0.00000             0.0
2 2025-12-17   NVDA  170.94        0.00000             0.0
1 2025-12-18   NVDA  174.14        0.00000             0.0
0 2025-12-19   NVDA  180.99       -0.08323             0.0
